**Step 1: Identify the task**
The task is to help the trailer carry goods from Nairobi to Kisumu on Kenya's highway network by:
- choosing the shortest open route
- make safe driving decision in regards to(traffic lights, weather, fuel level and pedestrians)

**Goal:** Arrive safely on time, with low fuel use while obeying traffic rules and speed limits

**Step 2: Assemble Relevant Knowledge**
The system needs to know:
- The action for each traffic light color
- The towns, the roads between them and their distances
- The speed limit, e.g. 80 km/h on highways and 65 km/h on regional roads.
- Only some towns have fuel stations

**Step 3: Define the vocabulary**
- **Predicates:** Light(color), Road(from, to, km), SpeedLimit(road), Escarpment(road), FuelStation(town), Weather(condition), Fuel(level), Obstacle(type)
- **Constants**
    Lights: Red, Yellow, Green
    Towns: Nairobi, Thika, Nyeri, Naivasha, Narok, Nakuru, Kericho, Eldoret, Kakamega, Kisumu
    Conditions: Clear, Rain, Low, Normal, Pedestrian, None

**Step 4: Encode General Knowledge about the domain**
Adding road network and local knowledge

In [1]:
rules = {
    "Red": "Stop — it’s unsafe to move.",
    "Yellow": "Slow down — prepare to stop.",
    "Green": "Go — it’s safe to proceed.",
}

# Road network
roads = {
    "Nairobi":  {"Thika": 45, "Naivasha": 91, "Narok": 143},
    "Thika":    {"Nyeri": 113},
    "Nyeri":    {"Nakuru": 167},
    "Naivasha": {"Nakuru": 70},
    "Narok":    {"Kericho": 157},
    "Nakuru":   {"Kericho": 107, "Eldoret": 156},
    "Kericho":  {"Kisumu": 84},
    "Eldoret":  {"Kakamega": 80},
    "Kakamega": {"Kisumu": 107},
    "Kisumu":   {},
}

# Local road knowledge
highways = [("Nairobi", "Thika"), ("Nairobi", "Naivasha"), ("Thika", "Nyeri"),
            ("Naivasha", "Nakuru"), ("Nakuru", "Eldoret"), ("Eldoret", "Kakamega"),
            ("Kakamega", "Kisumu")]                 # 'A' roads; all others are regional
escarpments = [("Nairobi", "Naivasha")]             # Rift Valley escarpment
fuel_stations = ["Nairobi", "Naivasha", "Nakuru", "Kericho", "Eldoret", "Kisumu"]

def speed_limit(a, b):
    return 80 if (a, b) in highways else 65

**Driving Rules**

Where each rule checks what the trailer sees, together with what the KB knows about the road or town and returns an action

In [2]:
def driving_rules(town, next_town, weather, fuel, obstacle):
    actions = []

    if obstacle == "Pedestrian":
        actions.append("Brake and sound horn - pedestrian ahead.")

    if fuel == "Low" and town in fuel_stations:
        actions.append(f"Refuel now at {town}.")

    if (town, next_town) in escarpments:
        actions.append("Escarpments ahead - use low gear, max 40 km/h")

    if weather == "Rain":
        actions.append("Wet road - reduce speed by 20 km/h and turn on headlights")
    return actions

**Step 5: Add new facts to the knowledge base**

We added `close_road()` so that agent can be told about a road closure

In [4]:
def show_kb():
    print("\n-- Current Knowledge Base --")
    for light, action in rules.items():
        print(f"{light:10} -> {action}")
    print("roads: ")

    for a, nexts in roads.items():
        for b, km in nexts.items():
            print(f" {a:9} -> {b:9} {km:4} km limit {speed_limit(a, b)} km/h")
    print("Fuel stations:", ",".join(fuel_stations))
    print("------")

def add_new_fact(color, action):
    """Add a new fact directly to the KB (- without user input)."""
    rules[color.capitalize()] = action
    print(f"Added fact: '{color.capitalize()}' -> '{action}'")

def close_road(a, b):
    """Tell the KB a road is closed (e.g. an accident)."""
    roads[a].pop(b, None)
    print(f"Road {a} -> {b} is now closed")

def add_fact():
    """Add new knowledge (facts) to the KB."""
    new_light = input("Enter new traffic light color: ").capitalize()
    new_action = input(f"What should the agent do when light is {new_light}?")
    rules[new_light] = new_action
    print(f"New fact added: '{new_light}' -> '{new_action}'")

add_fact()

# Viewing updated KB
show_kb()

New fact added: 'Red' -> 'stop'

-- Current Knowledge Base --
Red        -> stop
Yellow     -> Slow down — prepare to stop.
Green      -> Go — it’s safe to proceed.
           -> Red
roads: 
 Nairobi   -> Thika       45 km limit 80 km/h
 Nairobi   -> Naivasha    91 km limit 80 km/h
 Nairobi   -> Narok      143 km limit 65 km/h
 Thika     -> Nyeri      113 km limit 80 km/h
 Nyeri     -> Nakuru     167 km limit 65 km/h
 Naivasha  -> Nakuru      70 km limit 80 km/h
 Narok     -> Kericho    157 km limit 65 km/h
 Nakuru    -> Kericho    107 km limit 65 km/h
 Nakuru    -> Eldoret    156 km limit 80 km/h
 Kericho   -> Kisumu      84 km limit 65 km/h
 Eldoret   -> Kakamega    80 km limit 80 km/h
 Kakamega  -> Kisumu     107 km limit 80 km/h
Fuel stations: Nairobi,Naivasha,Nakuru,Kericho,Eldoret,Kisumu
------


**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers**

The agent will plan its route with UCS then reads the sensors and asks the KB what to do

In [5]:
import heapq

def ucs(start, goal):
    """Uniform Cost Search over the roads in the KB."""
    frontier = [(0, [start])]
    visited = set()

    while frontier:
        cost, path = heapq.heappop(frontier)
        town = path[-1]
        if town == goal:
            return path, cost
        if town in visited:
            continue
        visited.add(town)
        for nxt, km in roads[town].items():
            heapq.heappush(frontier, (cost+km, path+[nxt]))
    return None, None

def traffic_advice(light):
    action = rules.get(light, "Unknown signal - proceed with caution or stop")
    return action

In [6]:
# Current observation from sensors/camera
current_light = input("Enter the current traffic light color: ").capitalize()
weather = input("Weather (Clear/Rain): ").capitalize()
fuel = input("Fuel Level (Normal/Low): ").capitalize()
obstacle = input("Obstacle (None/Pedestrian): ").capitalize()

# Agent queries the knowledge base
path, km = ucs("Nairobi", "Kisumu")
actions = [traffic_advice(current_light)]+driving_rules("Nairobi", path[1], weather, fuel, obstacle)

# Outputting the result
print("\n ---Self Driving Trailer Decision System ---")
print(f"Route: {'->'.join(path)} ({km} km)")
print(f"Observed: light={current_light}, weather={weather}, fuel={fuel}, obstacle={obstacle}")

print("Agent decisions:")
for a in actions:
    print(" -", a)


 ---Self Driving Trailer Decision System ---
Route: Nairobi->Naivasha->Nakuru->Kericho->Kisumu (352 km)
Observed: light=Green, weather=Rain, fuel=Low, obstacle=None
Agent decisions:
 - Go — it’s safe to proceed.
 - Refuel now at Nairobi.
 - Escarpments ahead - use low gear, max 40 km/h
 - Wet road - reduce speed by 20 km/h and turn on headlights


The trailer drives town by town and at each town it uses the sensor readings in `journey` to decide what to do

In [7]:
# What sensors see at each town(light, weather, fuel, obstacle)

journey = {
    "Nairobi":  ("Red",   "Clear", "Normal", "None"),
    "Naivasha": ("Green", "Rain",  "Normal", "None"),
    "Nakuru":   ("Green", "Clear", "Normal", "Pedestrian"),
    "Kericho":  ("Green", "Clear", "Low",    "None"),
}

for a, b in zip(path, path[1:]):
    light, weather, fuel, obstacle = journey[a]

    print(f"At {a} (light {light}, {weather}, fuel {fuel}, obstacle {obstacle}):")
    print(" -", traffic_advice(light))
    for action in driving_rules(a, b, weather, fuel, obstacle):
        print(" -", action)
    print(f"  Drive to {b}: {roads[a][b]} km, limit {speed_limit(a, b)} km/h\n")
print("Arrived at Kisumu")

At Nairobi (light Red, Clear, fuel Normal, obstacle None):
 - stop
 - Escarpments ahead - use low gear, max 40 km/h
  Drive to Naivasha: 91 km, limit 80 km/h

At Naivasha (light Green, Rain, fuel Normal, obstacle None):
 - Go — it’s safe to proceed.
 - Wet road - reduce speed by 20 km/h and turn on headlights
  Drive to Nakuru: 70 km, limit 80 km/h

At Nakuru (light Green, Clear, fuel Normal, obstacle Pedestrian):
 - Go — it’s safe to proceed.
 - Brake and sound horn - pedestrian ahead.
  Drive to Kericho: 107 km, limit 65 km/h

At Kericho (light Green, Clear, fuel Low, obstacle None):
 - Go — it’s safe to proceed.
 - Refuel now at Kericho.
  Drive to Kisumu: 84 km, limit 65 km/h

Arrived at Kisumu


**Step 8: Querying the knowledge base**

In [8]:
print("Agent decision for 'Flashing':", traffic_advice("Flashing"))
print("Agent decision for 'Teal':", traffic_advice("Teal"))
print("Speed limit Nairobi->Naivasha:", speed_limit("Nairobi", "Naivasha"), "km/h")
print("Can the trailer refuel at Thika?", "Thika" in fuel_stations)
print("Roads leaving Nakuru:", roads["Nakuru"])

# Agent adapts when told about road closure
close_road("Naivasha", "Nakuru")
path2, km2 = ucs("Nairobi", "Kisumu")
print(f"New route: {'->'.join(path2)} ({km2} km)")
roads["Naivasha"]["Nakuru"] = 70 #re-open the road

Agent decision for 'Flashing': Unknown signal - proceed with caution or stop
Agent decision for 'Teal': Unknown signal - proceed with caution or stop
Speed limit Nairobi->Naivasha: 80 km/h
Can the trailer refuel at Thika? False
Roads leaving Nakuru: {'Kericho': 107, 'Eldoret': 156}
Road Naivasha -> Nakuru is now closed
New route: Nairobi->Narok->Kericho->Kisumu (384 km)


**Step 9: Debugging the Knowledge Base**


In [9]:
# Testing new or faulty signals to see agent's reaction
test_cases = ["Red", "Yellow", "Green", "Blue", "None"]

print("--Debugging the traffic lights--")
for color in test_cases:
    print(f"Light: {color:6} -> Action: {traffic_advice(color)}")

# Low fuel in a town without a fuel station, no rule fires
print("\nLow fuel in Thika ->", driving_rules("Thika", "Nyeri", "Clear", "Low", "None"))
print("GAP: the agent should be told to refuel at the next station")

# A weather the KB doesn't know about, no rule fires
print("Fog on the road ->", driving_rules("Kericho", "Kisumu", "Fog", "Normal", "None"))
print("GAP: the agent would not slow down because there is no rule for fog")

--Debugging the traffic lights--
Light: Red    -> Action: stop
Light: Yellow -> Action: Slow down — prepare to stop.
Light: Green  -> Action: Go — it’s safe to proceed.
Light: Blue   -> Action: Unknown signal - proceed with caution or stop
Light: None   -> Action: Unknown signal - proceed with caution or stop

Low fuel in Thika -> []
GAP: the agent should be told to refuel at the next station
Fog on the road -> []
GAP: the agent would not slow down because there is no rule for fog


**Final Reflection**

**Challenges**
- The 65 km/h limit on regional roads was an assumption
- Debuggin showed that the agent does nothing for situations it has no rule for

**Recommended Improvements**
- We could give rules priority so that to enable for example breaking for a pedestrian if they come before a green light